In [22]:
# Import libraries

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_score, recall_score, f1_score
)

from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score
from sklearn.base import clone
import xgboost as xgb

In [23]:
# Locate the project directory
project_root = Path.cwd()

if project_root.name == "scripts":
    project_root = project_root.parent

# Load cleaned master dataset
data_path = (
    project_root
    / "processed"
    / "stage2_clean_join"
    / "master_county_clean.csv"
)

master = pd.read_csv(data_path, dtype={"county_fips": str})

print("Dataset shape:", master.shape)

display(master.head())

Dataset shape: (3154, 38)


,county_fips,state_fips,state_abbr,state_name,county_name,centroid_longitude,centroid_latitude,commercial_electricity_rates_dollar_per_kwh_mean,commercial_electricity_rates_dollar_per_kwh_median,composite_siting_suitability_mean,...,state_siting_region_majority,water_cooling_fraction_mean,water_cooling_fraction_median,cerf_join,atlas_record_count,has_existing_data_center,atlas_join,ncsl_bill_count,ncsl_enacted_count,ncsl_join
0,01001,1,AL,ALABAMA,AUTAUGA,-86.642761,32.534912,0.056590,0.056590,0.019192,...,1.0,0.807202,1.0,both,0,NaN,left_only,17.0,8.0,both
1,01003,1,AL,ALABAMA,BALDWIN,-87.749781,30.660276,0.058395,0.058395,0.021779,...,1.0,0.000000,0.0,both,0,NaN,left_only,17.0,8.0,both
2,01005,1,AL,ALABAMA,BARBOUR,-85.393431,31.869549,0.054066,0.054066,0.006813,...,1.0,0.108687,0.0,both,0,NaN,left_only,17.0,8.0,both
3,01007,1,AL,ALABAMA,BIBB,-87.126463,32.998615,0.055336,0.055336,0.010149,...,1.0,1.000000,1.0,both,0,NaN,left_only,17.0,8.0,both
4,01009,1,AL,ALABAMA,BLOUNT,-86.567545,33.980817,0.053852,0.053852,0.019309,...,1.0,1.000000,1.0,both,0,NaN,left_only,17.0,8.0,both


In [24]:
# Check potential target values and Atlas coverage

print("Existing data center values:")
print(master["has_existing_data_center"].value_counts(dropna=False))

print("\nAtlas join values:")
print(master["atlas_join"].value_counts(dropna=False))

print("\nAtlas records by join status:")
display(
    master.groupby("atlas_join")["atlas_record_count"]
    .agg(["count", "min", "max"])
)

Existing data center values:
has_existing_data_center
NaN    2909
1.0     245
Name: count, dtype: int64

Atlas join values:
atlas_join
left_only    2909
both          245
Name: count, dtype: int64

Atlas records by join status:


,count,min,max
atlas_join,,,
both,245,1,213
left_only,2909,0,0


In [25]:
# Define target: whether a county has an Atlas record

master["target_atlas_presence"] = (
    master["atlas_record_count"] > 0
).astype(int)

print("Target distribution:")
print(master["target_atlas_presence"].value_counts())

print("\nTarget percentages:")
print(
    master["target_atlas_presence"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Target distribution:
target_atlas_presence
0    2909
1     245
Name: count, dtype: int64

Target percentages:
target_atlas_presence
0    92.23
1     7.77
Name: proportion, dtype: float64


In [26]:
# Define supervised modeling features

cerf_features = [
    "commercial_electricity_rates_dollar_per_kwh_mean",
    "industrial_electricity_rates_dollar_per_kwh_mean",
    "composite_siting_suitability_mean",
    "distance_to_substation_mean",
    "distance_to_substation_min",
    "land_value_dollar_per_sqft_mean",
    "mechanical_cooling_fraction_mean",
    "personal_property_tax_rate_mean",
    "real_property_tax_rate_mean",
    "sales_tax_rate_mean",
    "water_cooling_fraction_mean"
]

policy_features = [
    "ncsl_bill_count",
    "ncsl_enacted_count"
]

# Keep counties with complete CERF data
model_data = master.dropna(subset=cerf_features).copy()

# Define baseline and augmented features
X_baseline = model_data[cerf_features]
X_augmented = model_data[cerf_features + policy_features]
y = model_data["target_atlas_presence"]

print("Counties available for modeling:", len(model_data))
print("Baseline features:", X_baseline.shape[1])
print("Augmented features:", X_augmented.shape[1])

print("\nTarget distribution after filtering:")
print(y.value_counts())

print("\nMissing policy values:")
print(X_augmented[policy_features].isna().sum())

Counties available for modeling: 3099
Baseline features: 11
Augmented features: 13

Target distribution after filtering:
target_atlas_presence
0    2854
1     245
Name: count, dtype: int64

Missing policy values:
ncsl_bill_count       0
ncsl_enacted_count    0
dtype: int64


In [27]:
# Train/test split

train_idx, test_idx = train_test_split(
    model_data.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Baseline features
X_train_base = X_baseline.loc[train_idx]
X_test_base = X_baseline.loc[test_idx]

# Augmented features
X_train_aug = X_augmented.loc[train_idx]
X_test_aug = X_augmented.loc[test_idx]

# Target
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

print("Training counties:", len(train_idx))
print("Testing counties:", len(test_idx))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training counties: 2479
Testing counties: 620

Training target distribution:
target_atlas_presence
0    2283
1     196
Name: count, dtype: int64

Testing target distribution:
target_atlas_presence
0    571
1     49
Name: count, dtype: int64


# Logistic Regression Model

In [28]:
# Logistic Regression pipeline

logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

print(logistic_pipeline)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])


In [ ]:
# Logistic Regression tuning and cross-validation

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

param_grid = {
    "model__C": [0.01, 0.1, 1, 10]
}

logistic_results = {}
logistic_searches = {}

for name, X_train in {
    "Baseline": X_train_base,
    "Augmented": X_train_aug
}.items():

    search = GridSearchCV(
        estimator=logistic_pipeline,
        param_grid=param_grid,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    logistic_searches[name] = search

    logistic_results[name] = {
        "Model": name,
        "Best C": search.best_params_["model__C"],
        "CV Average Precision Mean": search.best_score_,
        "CV Average Precision Std": search.cv_results_[
            "std_test_score"
        ][search.best_index_]
    }

display(pd.DataFrame(logistic_results.values()).round(4))

/opt/homebrew/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/homebrew/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/homebrew/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/homebrew/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/homebrew/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/homebrew/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid

,Model,Best C,CV Average Precision Mean,CV Average Precision Std
0,Baseline,10,0.3291,0.0377
1,Augmented,10,0.3266,0.0393


In [39]:

# Examine which features are associated with Atlas data center presence
# using the best baseline Logistic Regression model

best_logistic = logistic_searches["Baseline"].best_estimator_
logistic_model = best_logistic.named_steps["model"]

logistic_coefficients = pd.DataFrame({
    "Feature": X_train_base.columns,
    "Coefficient": logistic_model.coef_[0]
})

logistic_coefficients["Direction"] = np.where(
    logistic_coefficients["Coefficient"] > 0,
    "Higher likelihood",
    "Lower likelihood"
)

logistic_coefficients = logistic_coefficients.sort_values(
    "Coefficient", ascending=False
)

display(logistic_coefficients.round(4))


,Feature,Coefficient,Direction
5,land_value_dollar_per_sqft_mean,7.2412,Higher likelihood
2,composite_siting_suitability_mean,0.3715,Higher likelihood
8,real_property_tax_rate_mean,0.1318,Higher likelihood
0,commercial_electricity_rates_dollar_per_kwh_mean,0.0611,Higher likelihood
6,mechanical_cooling_fraction_mean,0.0249,Higher likelihood
7,personal_property_tax_rate_mean,-0.0233,Lower likelihood
10,water_cooling_fraction_mean,-0.0249,Lower likelihood
3,distance_to_substation_mean,-0.0624,Lower likelihood
9,sales_tax_rate_mean,-0.1829,Lower likelihood
1,industrial_electricity_rates_dollar_per_kwh_mean,-0.8470,Lower likelihood


In [ ]:
# KNN tuning and cross-validation

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

knn_param_grid = {
    "model__n_neighbors": [5, 11, 21, 41],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

knn_results = {}
knn_searches = {}

for name, X_train in {
    "Baseline": X_train_base,
    "Augmented": X_train_aug
}.items():

    search = GridSearchCV(
        estimator=knn_pipeline,
        param_grid=knn_param_grid,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    knn_searches[name] = search

    knn_results[name] = {
        "Model": name,
        "Best Parameters": search.best_params_,
        "CV Average Precision Mean": search.best_score_,
        "CV Average Precision Std": search.cv_results_[
            "std_test_score"
        ][search.best_index_]
    }

display(pd.DataFrame(knn_results.values()).round(4))

,Model,Best Parameters,CV Average Precision Mean,CV Average Precision Std
0,Baseline,"{'model__n_neighbors': 41, 'model__p': 1, 'mod...",0.2761,0.0443
1,Augmented,"{'model__n_neighbors': 21, 'model__p': 1, 'mod...",0.2533,0.0497


In [31]:
# XGBoost tuning and cross-validation

def xgb_ap_scorer(estimator, X, y):
    probabilities = estimator.predict_proba(X)[:, 1]
    return average_precision_score(y, probabilities)

xgb_pipeline = Pipeline([
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    ))
])

xgb_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [2, 4],
    "model__learning_rate": [0.05, 0.1],
    "model__scale_pos_weight": [
        1,
        y_train.value_counts()[0] / y_train.value_counts()[1]
    ]
}

xgb_results = {}
xgb_searches = {}

for name, X_train in {
    "Baseline": X_train_base,
    "Augmented": X_train_aug
}.items():

    search = GridSearchCV(
        estimator=xgb_pipeline,
        param_grid=xgb_param_grid,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        error_score="raise"

    )

    search.fit(X_train, y_train)

    xgb_searches[name] = search

    xgb_results[name] = {
        "Model": name,
        "Best Parameters": search.best_params_,
        "CV Average Precision Mean": search.best_score_,
        "CV Average Precision Std": search.cv_results_[
            "std_test_score"
        ][search.best_index_]
    }

display(pd.DataFrame(xgb_results.values()).round(4))

,Model,Best Parameters,CV Average Precision Mean,CV Average Precision Std
0,Baseline,"{'model__learning_rate': 0.05, 'model__max_dep...",0.3539,0.0576
1,Augmented,"{'model__learning_rate': 0.05, 'model__max_dep...",0.3546,0.0479


In [32]:
# Compare supervised models

comparison = pd.concat([
    pd.DataFrame(logistic_results.values()).assign(Algorithm="Logistic Regression"),
    pd.DataFrame(knn_results.values()).assign(Algorithm="KNN"),
    pd.DataFrame(xgb_results.values()).assign(Algorithm="XGBoost")
], ignore_index=True)

comparison = comparison[
    ["Algorithm", "Model", "CV Average Precision Mean", "CV Average Precision Std"]
].sort_values("CV Average Precision Mean", ascending=False)

display(comparison.round(4))

,Algorithm,Model,CV Average Precision Mean,CV Average Precision Std
5,XGBoost,Augmented,0.3546,0.0479
4,XGBoost,Baseline,0.3539,0.0576
0,Logistic Regression,Baseline,0.3291,0.0377
1,Logistic Regression,Augmented,0.3266,0.0393
2,KNN,Baseline,0.2761,0.0443
3,KNN,Augmented,0.2533,0.0497


Best model: XGBoost Augmented — 0.3546 ± 0.0479.

Second: XGBoost Baseline — 0.3539 ± 0.0576.

Policy feature impact: Very small improvement for XGBoost, but reduced performance for Logistic Regression and KNN.

Conclusion so far: XGBoost leads, but we don't yet have convincing evidence that adding the two policy features meaningfully improves performance.

In [ ]:
# Evaluate tuned models on held-out test data

test_results = []

model_searches = {
    "Logistic Regression": logistic_searches,
    "KNN": knn_searches,
    "XGBoost": xgb_searches
}

for algorithm, searches in model_searches.items():
    for name, search in searches.items():
        X_test = X_test_base if name == "Baseline" else X_test_aug

        probabilities = search.best_estimator_.predict_proba(X_test)[:, 1]
        predictions = (probabilities >= 0.5).astype(int)

        test_results.append({
            "Algorithm": algorithm,
            "Model": name,
            "Average Precision": average_precision_score(y_test, probabilities),
            "ROC-AUC": roc_auc_score(y_test, probabilities),
            "Brier Score": brier_score_loss(y_test, probabilities),
            "Precision": precision_score(y_test, predictions, zero_division=0),
            "Recall": recall_score(y_test, predictions, zero_division=0),
            "F1": f1_score(y_test, predictions, zero_division=0)
        })

test_comparison = pd.DataFrame(test_results).sort_values(
    "Average Precision", ascending=False
)

display(test_comparison.round(4))

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,Algorithm,Model,Average Precision,ROC-AUC,Brier Score,Precision,Recall,F1
0,Logistic Regression,Baseline,0.4636,0.8517,0.1705,0.2552,0.7551,0.3814
1,Logistic Regression,Augmented,0.4596,0.8514,0.1714,0.2303,0.7143,0.3483
5,XGBoost,Augmented,0.4580,0.8558,0.1546,0.2353,0.7347,0.3564
4,XGBoost,Baseline,0.4460,0.8544,0.1549,0.2338,0.7347,0.3547
3,KNN,Augmented,0.3838,0.8171,0.0610,0.7500,0.0612,0.1132
2,KNN,Baseline,0.3668,0.8215,0.0627,0.0000,0.0000,0.0000


### Supervised Model Findings

I finished running Logistic Regression, KNN, and XGBoost using the CERF data, then ran each model again with the NCSL policy features added.

XGBoost performed the best during five fold cross validation, with an Average Precision of 0.3546 for the augmented model. However, when I evaluated the models on the test data, the baseline Logistic Regression model performed slightly better, with an Average Precision of 0.4636 compared to 0.4580 for augmented XGBoost.

KNN had the lowest Average Precision of the three models and struggled to identify counties with recorded data centers using the default prediction threshold. This suggests that comparing counties based on similar characteristics was less effective than the other approaches I tested.

One thing I noticed was that adding the policy features did not consistently improve the results. XGBoost improved slightly, but Logistic Regression and KNN performed worse during cross validation. This suggests that the two policy features did not provide a consistent predictive benefit across the models.

I also ran into an issue with XGBoost where all 16 configurations failed to return valid scores. After troubleshooting, I found that the problem was related to compatibility with scikit learn. Upgrading XGBoost resolved the issue and allowed the model tuning to run successfully.

I then examined feature importance, removed selected features to see how performance changed, tested different classification thresholds, and reviewed counties that the models predicted incorrectly. Land value stood out as an important feature, and removing it reduced XGBoost's performance. The threshold analysis also showed how changing the classification cutoff affected precision and recall.

Overall, XGBoost was the strongest model based on cross validation, while Logistic Regression performed slightly better on the held out test data. The results were close enough that I would want to evaluate the models further before selecting one for practical use.

An important limitation is that the models predict whether a county has a data center recorded in the Atlas dataset, not whether a new data center will be built there. The results are useful for understanding relationships between county characteristics and recorded data center locations, but they should not be interpreted as forecasts of future development.

In [ ]:
# XGBoost augmented feature importance

best_xgb = xgb_searches["Augmented"].best_estimator_

importance = permutation_importance(
    best_xgb,
    X_train_aug,
    y_train,
    scoring="average_precision",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_train_aug.columns,
    "Importance Mean": importance.importances_mean,
    "Importance Std": importance.importances_std
}).sort_values("Importance Mean", ascending=False)

display(importance_df.round(4))

,Feature,Importance Mean,Importance Std
5,land_value_dollar_per_sqft_mean,0.2996,0.0091
2,composite_siting_suitability_mean,0.0727,0.0218
3,distance_to_substation_mean,0.0508,0.0086
8,real_property_tax_rate_mean,0.0448,0.0103
9,sales_tax_rate_mean,0.0331,0.0094
1,industrial_electricity_rates_dollar_per_kwh_mean,0.0135,0.0050
0,commercial_electricity_rates_dollar_per_kwh_mean,0.0085,0.0051
11,ncsl_bill_count,0.0031,0.0031
12,ncsl_enacted_count,0.0029,0.0004
4,distance_to_substation_min,0.0019,0.0035


Land value is clearly the most influential feature in this particular model. The CERF suitability score, average distance to substations, and tax rates also contribute to its predictions.
The two NCSL policy features have relatively low importance, which is consistent with our earlier finding that adding policy data produced only a small improvement in XGBoost's Average Precision.

In [ ]:
# XGBoost feature ablation

ablation_sets = {
    "All 13 features": cerf_features + policy_features,

    "Without land value": [
        f for f in cerf_features + policy_features
        if f != "land_value_dollar_per_sqft_mean"
    ],

    "Without policy features": cerf_features,

    "Without top 3 CERF features": [
        f for f in cerf_features + policy_features
        if f not in [
            "land_value_dollar_per_sqft_mean",
            "composite_siting_suitability_mean",
            "distance_to_substation_mean"
        ]
    ]
}

ablation_results = []

for name, features in ablation_sets.items():

    model = clone(xgb_searches["Augmented"].best_estimator_)

    scores = cross_val_score(
        model,
        X_train_aug[features],
        y_train,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        error_score="raise"
    )

    ablation_results.append({
        "Feature Set": name,
        "Features": len(features),
        "CV AP Mean": scores.mean(),
        "CV AP Std": scores.std()
    })

ablation_df = pd.DataFrame(ablation_results)

display(ablation_df.round(4))

,Feature Set,Features,CV AP Mean,CV AP Std
0,All 13 features,13,0.3546,0.0479
1,Without land value,12,0.2541,0.0549
2,Without policy features,11,0.3539,0.0576
3,Without top 3 CERF features,10,0.1434,0.0246


1. Land value is particularly important.
Removing land_value_dollar_per_sqft_mean reduced Average Precision from 0.3546 to 0.2541. This is a substantial decrease and supports our earlier permutation importance findings.
2. The policy features contribute very little to predictive performance.
Removing ncsl_bill_count and ncsl_enacted_count reduced Average Precision by only 0.0007.
This is consistent with our earlier results showing that adding policy features provided almost no improvement during cross validation.
3. The top three CERF features collectively matter a lot.
Removing land value, composite siting suitability, and average distance to substations reduced Average Precision to 0.1434.
This suggests that the model relies heavily on information contained in these features.

In [ ]:
# Classification threshold sensitivity analysis

best_xgb = xgb_searches["Augmented"].best_estimator_

cv_probabilities = cross_val_predict(
    best_xgb,
    X_train_aug,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

threshold_results = []

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    predictions = (cv_probabilities >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, predictions, zero_division=0),
        "Recall": recall_score(y_train, predictions, zero_division=0),
        "F1": f1_score(y_train, predictions, zero_division=0),
        "Predicted Positive": predictions.sum()
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(4))

,Threshold,Precision,Recall,F1,Predicted Positive
0,0.1,0.0804,0.9898,0.1487,2413
1,0.2,0.0891,0.9337,0.1627,2054
2,0.3,0.1200,0.8010,0.2088,1308
3,0.4,0.1712,0.7092,0.2758,812
4,0.5,0.2094,0.5918,0.3093,554
5,0.6,0.2884,0.4694,0.3573,319
6,0.7,0.3711,0.3673,0.3692,194
7,0.8,0.4800,0.3061,0.3738,125
8,0.9,0.5385,0.0714,0.1261,26


In [37]:
# Identify misclassified counties

best_xgb = xgb_searches["Augmented"].best_estimator_

test_probabilities = best_xgb.predict_proba(X_test_aug)[:, 1]
test_predictions = (test_probabilities >= 0.5).astype(int)

errors = model_data.loc[test_idx, [
    "county_fips", "atlas_record_count"
]].copy()

errors["Actual"] = y_test
errors["Predicted"] = test_predictions
errors["Probability"] = test_probabilities

errors["Error Type"] = np.select(
    [
        (errors["Actual"] == 0) & (errors["Predicted"] == 1),
        (errors["Actual"] == 1) & (errors["Predicted"] == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

misclassified = errors[errors["Error Type"] != "Correct"]

print("Misclassified counties:", len(misclassified))
print(misclassified["Error Type"].value_counts())

display(
    misclassified.sort_values(
        "Probability", ascending=False
    ).head(10)
)


Misclassified counties: 130
Error Type
False Positive    117
False Negative     13
Name: count, dtype: int64


,county_fips,atlas_record_count,Actual,Predicted,Probability,Error Type
1866,36059,0,0,1,0.934233,False Positive
2275,42045,0,0,1,0.918159,False Positive
2941,51650,0,0,1,0.880294,False Positive
235,06087,0,0,1,0.877523,False Positive
247,06111,0,0,1,0.867024,False Positive
2939,51630,0,0,1,0.863129,False Positive
999,20209,0,0,1,0.861571,False Positive
361,12071,0,0,1,0.851954,False Positive
380,12109,0,0,1,0.851462,False Positive
1792,34019,0,0,1,0.848885,False Positive


The model produces considerably more false positives than false negatives. This is consistent with the relatively high recall and low precision we observed in Cell 12.
For example, county FIPS 36059 received a predicted probability of 0.9342, even though it had no matching Atlas record.

In [38]:
# Examine specific misclassified counties

false_positives = misclassified[
    misclassified["Error Type"] == "False Positive"
].sort_values("Probability", ascending=False).head(2)

false_negatives = misclassified[
    misclassified["Error Type"] == "False Negative"
].sort_values("Probability").head(2)

selected_errors = pd.concat([
    false_positives,
    false_negatives
])

selected_features = [
    "land_value_dollar_per_sqft_mean",
    "composite_siting_suitability_mean",
    "distance_to_substation_mean",
    "ncsl_bill_count",
    "ncsl_enacted_count"
]

error_details = selected_errors.join(
    model_data[selected_features]
)

display(error_details.round(4))

,county_fips,atlas_record_count,Actual,Predicted,Probability,Error Type,land_value_dollar_per_sqft_mean,composite_siting_suitability_mean,distance_to_substation_mean,ncsl_bill_count,ncsl_enacted_count
1866,36059,0,0,1,0.9342,False Positive,1.1472,0.0037,16.0952,227.0,6.0
2275,42045,0,0,1,0.9182,False Positive,0.5384,0.0030,24.7858,42.0,1.0
856,19121,1,1,0,0.2296,False Negative,0.1414,0.0000,75.1966,21.0,2.0
1624,30039,1,1,0,0.2983,False Negative,0.0516,0.0000,173.1232,33.0,11.0


1. False positives
Counties 36059 and 42045 have higher land values and shorter average distances to substations than the two false negative examples.
These characteristics may help explain why XGBoost assigned them high probabilities, given our feature importance findings. However, we cannot confirm the exact reason from these values alone.
2. False negatives
Counties 19121 and 30039 have relatively low land values, zero composite suitability scores, and greater average distances to substations.
The model may be missing recorded data centers in counties that do not resemble the stronger positive patterns it learned.
3. Potential improvements
We can investigate whether the model is relying too heavily on land value, evaluate different classification thresholds, and examine whether additional geographic or infrastructure features improve performance.
We should also verify whether the false positives reflect incomplete Atlas coverage rather than genuinely incorrect predictions.

### Supervised Modeling Summary

I finished building and testing the three supervised models: Logistic Regression, KNN, and XGBoost. I tested each model using the 11 CERF features and then again with the two NCSL policy features added. I used cross validation to tune the models and a separate test dataset to evaluate the results.

**Model Results**

XGBoost performed the best during cross validation with an Average Precision of 0.3546. However, Logistic Regression had the highest Average Precision on the test data at 0.4636, compared to XGBoost at 0.4580.

KNN performed the worst overall. One thing I noticed is that adding the policy features did not consistently improve the models. XGBoost improved slightly, but the other models performed worse during cross validation.

**Feature Importance and Testing**

Land value was by far the most important feature for XGBoost, followed by composite siting suitability and distance to substations.

I also tested removing features to see how much they affected the model. Removing land value dropped Average Precision from 0.3546 to 0.2541. Removing the top three CERF features dropped it to 0.1434. Removing the policy features barely changed the results.

**Sensitivity and Model Errors**

I tested different prediction thresholds and found that 0.8 had the highest F1 score out of the thresholds tested. However, it also missed more counties with recorded data centers.

At the original 0.5 threshold, XGBoost made 130 incorrect predictions compared to the Atlas labels, with 117 false positives and 13 false negatives. I looked at four specific counties to better understand the errors. The model seemed to struggle more with counties that had lower land values and were farther from substations, although more analysis would be needed to confirm why.

**Issues and Overall Findings**

I ran into a compatibility issue with XGBoost that caused all 16 configurations to return invalid scores. After troubleshooting, I upgraded XGBoost, which fixed the issue.

Overall, XGBoost looks like the strongest model based on cross validation, but Logistic Regression was competitive on the test data. The biggest finding so far is that the CERF features, especially land value, seem to provide most of the predictive information, while the policy features added very little.

One limitation is that we are predicting whether counties have data centers recorded in the Atlas dataset, not necessarily whether data centers actually exist or will be built there.

The main modeling work is complete. I still need to check how well the model performs across different states, clean up the notebook, and pull the final findings into the report.